# Retrieval Is Filtering, Not Search: A Mental Model for Enterprise RAG

> 📖 **Full article on Towards Data Science:** [Retrieval Is Filtering, Not Search: A Mental Model for Enterprise RAG](https://towardsdatascience.com/retrieval-is-filtering-not-search-a-mental-model-for-enterprise-rag/)
>
> This is the **runnable, code-only companion**. The explanations, the diagrams and the *why* live in the article. Here you just run the pipeline end to end. Every section below links back to the matching part of the article. **To understand any step, read the article.**


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# Bootstrap: add the repo's src/ to sys.path so we can import docintel without pip install.
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name in ("notebooks", "book_1") else NOTEBOOK_DIR
SRC_PATH = str(REPO_ROOT)
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

In [ ]:
import re
import json
import io
import contextlib
import numpy as np
import pandas as pd
import fitz
from pydantic import BaseModel

from lib.core import get_embedding, embed_page_df
from lib.parsing.pdf import fitz_pdf_to_line_df, build_page_df
from lib.parsing.pdf.fitz import build_toc_df
from lib.retrieval import cosine_sim, retrieve_pages, retrieve_pages_by_similarity

# matplotlib is used in a few figure chunks (RRF formula, line-level heatmap).
# Imported here so any later chunk can use `plt` without re-importing.
import matplotlib.pyplot as plt

In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(
    api_key=os.getenv("API_KEY"),
    base_url=os.getenv("BASE_URL"),
)
model_chat = os.getenv("MODEL_CHAT", "gpt-4.1")
model_embed = os.getenv("MODEL_EMBED", "text-embedding-3-small")

In [ ]:
# Parse once, reload forever : `load_parsed` reads from output/storage.sqlite ;
# on miss, we parse the PDF and persist via `save_parsed`. Re-renders then
# skip the PDF parsing entirely.
from lib.storage.sqlite_store import load_parsed, save_parsed

pdf_path = REPO_ROOT / "data" / "paper" / "1706.03762v7.pdf"
doc = fitz.open(pdf_path)

_cached = load_parsed(pdf_path)
if _cached is not None and _cached.get("line_df") is not None:
    line_df = _cached["line_df"]
    page_df = _cached["page_df"]
    toc_df  = _cached["toc_df"]
else:
    line_df = fitz_pdf_to_line_df(pdf_path)
    page_df = build_page_df(line_df)
    toc_df  = build_toc_df(doc)
    save_parsed(pdf_path, {
        "line_df": line_df, "page_df": page_df, "toc_df": toc_df,
    })

# Convenience column expected by the rest of the notebook (start_page and end_page
# are computed by build_toc_df per the implicit-end convention from Article 5B).
toc_df["section_id"] = toc_df["toc_idx"].astype(str)

print(f"{len(line_df)} lines across {line_df['page_num'].nunique()} pages")
print(f"{len(toc_df)} TOC entries from native outline")
toc_df[["section_id", "level", "title", "start_page", "end_page"]]

In [ ]:
def attach_section_id(line_df: pd.DataFrame, toc_df: pd.DataFrame) -> pd.DataFrame:
    """Attach a section_id to each line based on the TOC's page ranges.

    When a sub-section shares pages with its parent, the deepest entry wins
    (last write in the loop). Lines before the first TOC entry stay None.
    """
    df = line_df.copy()
    df["section_id"] = None
    for _, sec in toc_df.iterrows():
        in_section = (df["page_num"] >= sec["start_page"]) & (df["page_num"] <= sec["end_page"])
        df.loc[in_section, "section_id"] = sec["section_id"]
    return df

line_df = attach_section_id(line_df, toc_df)
print(f"line_df: {len(line_df)} rows, {line_df['section_id'].nunique()} distinct section_ids")
line_df.head(3)[["page_num", "line_num", "section_id", "text"]]

In [ ]:
# Embed page_df once, up front, so any §1 / §3 / §4 chunk can run a cosine
# baseline without re-paying the API cost. Cached cross-document and
# cross-article in `output/storage.sqlite` keyed by sha256(model + text), so
# the same text embedded by any other article hits the cache for free.
from lib.storage.sqlite_store import embed_texts_cached
_embed_model = os.getenv("MODEL_EMBED", "text-embedding-ada-002")
page_df["embedding"] = embed_texts_cached(
    client, page_df["text"].tolist(), model=_embed_model,
)

## 1. Retrieval as filtering on structured tables

📖 **Read section 1 of the article on TDS:** [Retrieval Is Filtering, Not Search: A Mental Model for Enterprise RAG](https://towardsdatascience.com/retrieval-is-filtering-not-search-a-mental-model-for-enterprise-rag/)


In [ ]:
# Side-by-side: the dense line_df vs the sparse toc_df.
print(f"line_df: {len(line_df):,} rows")
display(line_df.head(5)[["page_num", "line_num", "section_id", "text"]])
print(f"\ntoc_df: {len(toc_df)} rows")
display(toc_df[["section_id", "level", "title", "start_page", "end_page"]])

In [ ]:
# Sketch the (anchor scope, context scope) pairing each question type wants.
question_scope_pairings = pd.DataFrame([
    {"question_type": "needle in a haystack",       "example": "What is the policy number?",        "anchor_scope": "line (token match)",       "context_scope": "page (5 lines)"},
    {"question_type": "point lookup",               "example": "What is the annual premium?",        "anchor_scope": "line (keyword + amount)", "context_scope": "section (50-200 lines)"},
    {"question_type": "listing",                    "example": "What are all the obligations?",      "anchor_scope": "lines (multi)",           "context_scope": "section + cross-section (500-2000 lines)"},
    {"question_type": "scoped synthesis",           "example": "Summarize the warranty section.",    "anchor_scope": "section title in toc_df", "context_scope": "section body (200-1500 lines)"},
])
question_scope_pairings

## 2. Anchor and context: the two granularities

📖 **Read section 2 of the article on TDS:** [Retrieval Is Filtering, Not Search: A Mental Model for Enterprise RAG](https://towardsdatascience.com/retrieval-is-filtering-not-search-a-mental-model-for-enterprise-rag/)


In [ ]:
# A small dispatch table mapping question type to anchor / context scope.
scope_dispatch = pd.DataFrame([
    {"question_type": "field extraction",       "anchor": "line",          "context": "page"},
    {"question_type": "section retrieval",      "anchor": "section title", "context": "section body"},
    {"question_type": "conditional lookup",     "anchor": "line",          "context": "section"},
    {"question_type": "open-ended scoped",      "anchor": "section title", "context": "section + cross-section"},
    {"question_type": "open-ended corpus-wide", "anchor": "line",          "context": "paragraph + corpus filter"},
])
scope_dispatch

---

📖 **Read the full article on TDS:** [Retrieval Is Filtering, Not Search: A Mental Model for Enterprise RAG](https://towardsdatascience.com/retrieval-is-filtering-not-search-a-mental-model-for-enterprise-rag/)

The whole series ships on Towards Data Science: [all articles by Angela Shi](https://towardsdatascience.com/author/angela-shi/)

This notebook ships a runnable slice. **For the complete code** (every brick, the dispatcher, the schemas), get in touch on Ko-fi: https://ko-fi.com/angelashi
